# Hands-on Task 1: Expense Calculator Agent

## Objective

Build a cyclic LangGraph agent that performs financial calculations using tools.

## Features

- Add expenses
- Apply tax
- Split bill

## Example Queries

- What is the total of 1200 and 800?
- Add 2000 and 1500, apply 18% tax, and split among 3 people.

## Learning Goals

✅ Tool Calling

✅ Cyclic Agent Flows

✅ Agent ↔ Tool Loops

✅ Multi-Step Financial Computations

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    base_url="https://llmgw-wp.tekstac.com",
    temperature=0
)

In [3]:
from langchain_core.tools import tool

@tool
def add_expenses(a: float, b: float) -> float:
    """Add two expense values."""
    return a + b


@tool
def apply_tax(amount: float, tax_percent: float) -> float:
    """Apply tax percentage to amount."""
    return amount + (amount * tax_percent / 100)


@tool
def split_bill(amount: float, people: int) -> float:
    """Split bill among people."""
    return amount / people


tools = [
    add_expenses,
    apply_tax,
    split_bill
]

In [8]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    llm,
    tools
)

/tmp/ipykernel_12871/3306438697.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [9]:
response = agent.invoke(
    {
        "messages": [
            (
                "user",
                "What is the total of 1200 and 800?"
            )
        ]
    }
)

for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

What is the total of 1200 and 800?
================================== Ai Message ==================================

[{'id': 'toolu_bdrk_01J3VqK62rJJUWd3W7CNmBT5', 'caller': None, 'input': {'a': 1200, 'b': 800}, 'name': 'add_expenses', 'type': 'tool_use'}]
Tool Calls:
  add_expenses (toolu_bdrk_01J3VqK62rJJUWd3W7CNmBT5)
 Call ID: toolu_bdrk_01J3VqK62rJJUWd3W7CNmBT5
  Args:
    a: 1200
    b: 800
================================= Tool Message =================================
Name: add_expenses

2000.0
================================== Ai Message ==================================

The total of 1200 and 800 is **2000**.


In [10]:
response = agent.invoke(
    {
        "messages": [
            (
                "user",
                "Add 2000 and 1500, apply 18 percent tax, and split among 3 people."
            )
        ]
    }
)

for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Add 2000 and 1500, apply 18 percent tax, and split among 3 people.
================================== Ai Message ==================================

[{'id': 'toolu_bdrk_01DmN7HHTGRn1tyZsf8ChomM', 'caller': None, 'input': {'a': 2000, 'b': 1500}, 'name': 'add_expenses', 'type': 'tool_use'}]
Tool Calls:
  add_expenses (toolu_bdrk_01DmN7HHTGRn1tyZsf8ChomM)
 Call ID: toolu_bdrk_01DmN7HHTGRn1tyZsf8ChomM
  Args:
    a: 2000
    b: 1500
================================= Tool Message =================================
Name: add_expenses

3500.0
================================== Ai Message ==================================

[{'citations': None, 'text': "Now I'll apply the 18% tax to this amount:", 'type': 'text'}, {'id': 'toolu_bdrk_01K35xNvKC4ShBxxPwRTB3tL', 'caller': None, 'input': {'amount': 3500, 'tax_percent': 18}, 'name': 'apply_tax', 'type': 'tool_use'}]
Tool Calls:
  apply_tax (toolu_bdrk_01K35xNvKC4ShBxxP

# How the Loop Works

The ReAct agent follows this cycle:

User Request
      ↓
Agent Reasoning
      ↓
Select Tool
      ↓
Execute Tool
      ↓
Return Result
      ↓
Agent Reasoning Again
      ↓
More Tools Needed?
      ↓
YES → Loop Back
NO  → Final Response

For:

"Add 2000 and 1500, apply 18% tax, and split among 3 people"

The agent executes:

1. add_expenses(2000,1500)
2. apply_tax(3500,18)
3. split_bill(4130,3)

Then returns the final answer.

# Hands-on Task 2: Travel Planner with Memory

## Objective

Build a chatbot that remembers information across multiple conversations.

The bot should remember:

- Destination
- Travel Dates
- Travel Preferences

## Learning Goals

✅ MemorySaver

✅ Session Tracking

✅ thread_id

✅ Long-Term Context Retention

In [11]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_anthropic import ChatAnthropic
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

In [12]:
llm = ChatAnthropic(
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    base_url="https://llmgw-wp.tekstac.com",
    temperature=0
)

In [13]:
memory = MemorySaver()

agent = create_react_agent(
    llm,
    tools=[],
    checkpointer=memory
)

/tmp/ipykernel_12871/3961314064.py:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [14]:
config = {
    "configurable": {
        "thread_id": "traveller_001"
    }
}

In [15]:
response = agent.invoke(
    {
        "messages": [
            ("user", "I am planning a trip to Bali.")
        ]
    },
    config=config
)

response["messages"][-1].pretty_print()

================================== Ai Message ==================================

# Bali Trip Planning

That sounds exciting! Here are some key things to consider:

## Basics
- **Best time to visit:** April-October (dry season)
- **Visa:** Most nationalities get 30 days visa-free or can get a visa on arrival
- **Currency:** Indonesian Rupiah (IDR)
- **Language:** Indonesian, but English is widely spoken in tourist areas

## Popular Areas
- **Ubud** - cultural heart, rice terraces, arts
- **Seminyak/Canggu** - beaches, nightlife, restaurants
- **Sanur** - quieter beach town
- **Uluwatu** - cliffs, temples, surfing

## Must-See Attractions
- Temples (Tanah Lot, Besakih, Tirta Empul)
- Rice paddies and hiking
- Beaches and water sports
- Monkey Forest in Ubud

## Practical Tips
- Book accommodations in advance during peak season
- Rent a scooter or hire a driver
- Try local food and street markets
- Respect local customs and dress modestly at temples

## Questions to Help Me Assist Better

In [16]:
response = agent.invoke(
    {
        "messages": [
            ("user", "I prefer budget hotels.")
        ]
    },
    config=config
)

response["messages"][-1].pretty_print()

================================== Ai Message ==================================

# Budget Hotels in Bali

Great! Bali is very budget-friendly. Here's what you should know:

## Budget Hotel Options
- **$10-25/night:** Basic guesthouses and hostels
- **$25-50/night:** Mid-range budget hotels with good amenities
- **$50+/night:** Nicer budget hotels with pools, AC, hot water

## Best Areas for Budget Stays

**Ubud**
- Most affordable overall
- Guesthouses everywhere ($10-30/night)
- Great for backpackers and budget travelers

**Canggu/Seminyak**
- More expensive but still budget options exist
- Mix of hostels and budget hotels ($20-40/night)
- Better nightlife and beach access

**Sanur**
- Quieter, good value
- Budget hotels $15-35/night

**Kuta**
- Very touristy but budget-friendly
- Lots of hostels and cheap hotels

## Budget Hotel Tips
- Book on Agoda, Booking.com, or Hostelworld
- Negotiate prices directly (especially for longer stays)
- Check reviews carefully on budget properties
-

In [17]:
response = agent.invoke(
    {
        "messages": [
            ("user", "My trip is in December.")
        ]
    },
    config=config
)

response["messages"][-1].pretty_print()

================================== Ai Message ==================================

# Bali in December - Important Info

## Weather Warning ⚠️
December is **wet season** in Bali:
- Heavy rainfall, especially afternoons
- High humidity
- Occasional flooding in some areas
- Shorter daylight hours

**However:** It's also the **cheapest time to visit** - great for budget travelers!

## Advantages for Budget Travelers
- **Lower prices** on hotels, flights, and tours (30-50% cheaper)
- Fewer tourists
- Less crowded attractions
- Better deals on accommodations

## What to Pack
- Lightweight rain jacket or poncho
- Waterproof bag for electronics
- Quick-dry clothes
- Umbrella
- Still bring sunscreen (sun is strong between showers)

## Activity Adjustments
- Indoor activities: museums, cooking classes, spas
- Visit temples (often less crowded)
- Hiking early morning (before rain)
- Beach time between showers
- Water sports still possible but conditions vary

## Budget Hotel Considerations
- Rooms

In [18]:
response = agent.invoke(
    {
        "messages": [
            ("user", "Summarise my travel plan.")
        ]
    },
    config=config
)

response["messages"][-1].pretty_print()

================================== Ai Message ==================================

# Your Bali Travel Plan Summary

**Destination:** Bali, Indonesia

**Travel Time:** December (wet season)

**Accommodation:** Budget hotels ($10-50/night range)

**Key Preferences:**
- Budget-conscious travel
- Affordable accommodations
- Taking advantage of low-season prices

## What We've Covered
✓ Best areas for budget stays (Ubud, Canggu, Sanur, Kuta)
✓ Weather expectations (rainy season)
✓ Money-saving tips
✓ What to pack for wet season
✓ Activity suggestions despite rain

## Still To Confirm
- Exact dates in December?
- How many days/nights?
- Which area(s) to focus on?
- Specific interests (culture, beaches, adventure, relaxation)?
- Approximate total budget?

Once you provide these details, I can give you more specific recommendations for:
- Exact budget hotel suggestions
- Day-by-day itinerary
- Best transportation options
- Specific attractions to visit

Would you like to fill in any of these de

# How Memory Works

MemorySaver stores conversation state.

thread_id identifies a unique user session.

Example:

thread_id = "traveller_001"

All future calls using the same thread_id automatically
receive previous messages from memory.

This allows the assistant to remember:

- Destination = Bali
- Month = December
- Preference = Budget Hotels

without passing these values manually.

# Hands-on Task 3: Access Control Approval System

## Objective

Build a Human-in-the-Loop workflow.

Before any access request is executed:

1. Agent decides which tool to use
2. Workflow pauses
3. Human reviews action
4. User approves or rejects
5. Tool executes only if approved

## Learning Goals

✅ Human-in-the-Loop

✅ Enterprise Approval Flows

✅ interrupt_before

✅ Safe Tool Execution

In [19]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

In [20]:
llm = ChatAnthropic(
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    api_key=os.environ.get("ANTHROPIC_API_KEY"),
    base_url="https://llmgw-wp.tekstac.com",
    temperature=0
)

In [21]:
@tool
def grant_access(user: str, system: str):
    """Grant system access."""
    return f"Access granted to {user} for {system}"


@tool
def revoke_access(user: str, system: str):
    """Revoke system access."""
    return f"Access revoked for {user} from {system}"


tools = [
    grant_access,
    revoke_access
]

In [30]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

memory = MemorySaver()

agent = create_react_agent(
    llm,
    tools,
    checkpointer=memory,
    interrupt_before=["tools"]
)

graph = agent

/tmp/ipykernel_12871/4272548710.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [31]:
config = {
    "configurable": {
        "thread_id": "access_request"
    }
}

result = graph.invoke(
    {
        "messages": [
            (
                "user",
                "Give database admin access to user Kiran"
            )
        ]
    },
    config=config
)

In [32]:
snapshot = graph.get_state(config)

print("Next Node:")
print(snapshot.next)

Next Node:
('tools',)


In [35]:
from langgraph.types import Command

approval = input("Approve? (yes/no): ")

if approval.lower() == "yes":

    result = graph.invoke(
        Command(resume=True),
        config=config
    )

    for msg in result["messages"]:
        print(msg)

else:
    print("Access request rejected.")


content='Give database admin access to user Kiran' additional_kwargs={} response_metadata={} id='d8bfb8c4-0990-4e99-861a-60e45606848c'
content=[{'citations': None, 'text': "I'll grant database admin access to user Kiran.", 'type': 'text'}, {'id': 'toolu_bdrk_01UWTBsdLSL3n4EXJAt8zxVj', 'caller': None, 'input': {'user': 'Kiran', 'system': 'database admin'}, 'name': 'grant_access', 'type': 'tool_use'}] additional_kwargs={} response_metadata={'id': 'msg_bdrk_01HdYPVQUM8bEp9LhUhC9Xsf', 'container': None, 'model': 'global.anthropic.claude-haiku-4-5-20251001-v1:0', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': None, 'input_tokens': 643, 'output_tokens': 86, 'server_tool_use': None, 'service_tier': None, 'total_tokens': 729}} id='run--54c01e03-9349-478b-9831-955a410bbb82-0' tool_calls=[{'name': '

# Human-in-the-Loop Flow

User Request
      ↓
Agent Analysis
      ↓
Select Tool
      ↓
PAUSE (interrupt_before=["tools"])
      ↓
Human Reviews
      ↓
Approve?
   /      \
 YES      NO
  |        |
Execute   Stop
 Tool
  |
Result

Example:

User:
"Give database admin access to Kiran"

Agent decides:

grant_access(
    user="Kiran",
    system="database admin"
)

Execution pauses.

Manager Approval Required.

Only after approval does the tool execute.

This pattern is commonly used for:

- Access Provisioning
- Financial Transactions
- Production Deployments
- Compliance Workflows